# Importing Social Media Data

In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from h5n1.db import get_engine

In [ ]:
QUERY = "SELECT * FROM x_post_unique"
df = pd.read_sql(QUERY, get_engine())

In [ ]:
df.dtypes

In [ ]:
df

# EDA

* Total amount of language

In [ ]:
df_lan_filter = df[df['lang'].isin(['en', 'es', 'fr'])]

In [ ]:
sns.countplot(df_lan_filter['lang'])
plt.title('Count of posts in different languages')
plt.ylabel('Languages')
plt.show()

* Different post type

In [ ]:
sns.countplot(df['post_type'])
plt.title('Count of different post types')
plt.ylabel('Post types')
plt.show()

* Type of user location

In [ ]:
unique = set(df['user_location'].values)
print(unique)

* Different source query

In [ ]:
df_no_subtypes =  df.loc[(df['source_query'] != 'subtypes')&(df['source_query'] != 'subtypes_and_rest')]

In [ ]:
sns.countplot(df_no_subtypes['source_query'], order=df_no_subtypes['source_query'].value_counts().index)
plt.title('Count of different source queries')
plt.ylabel('Source Query')
plt.show()

* Investigation of relationship between followers and impressions/retweets/replies/favorites 

In [ ]:
sns.scatterplot(x=df['followers'], y=df['impressions'])
plt.title('Relationship between followers and impressions')
plt.show()

In [ ]:
sns.scatterplot(x=df['followers'], y=df['retweets'])
plt.title('Relationship between followers and retweets')
plt.show()

In [ ]:
sns.scatterplot(x=df['followers'], y=df['replies'])
plt.title('Relationship between followers and replies')
plt.show()

In [ ]:
sns.scatterplot(x=df['followers'], y=df['favorites'])
plt.title('Relationship between followers and favorites')
plt.show()

In [ ]:
QUERY = "SELECT * FROM x_post_analysis"
df1 = pd.read_sql(QUERY, get_engine())

In [ ]:
df1

In [ ]:
df1 = df1.drop_duplicates(subset='post_id', keep='first')

In [ ]:
len(df1)

In [ ]:
top_n = df1['search_terms'].value_counts().iloc[:20].index

plt.figure(figsize=(10, 10))
sns.countplot(df1['search_terms'], order=top_n)
plt.title('Count of Top 20 Search Terms Posts')
plt.ylabel('Search Terms')
plt.show()

In [ ]:
df1['disposition'].describe()

In [ ]:
print(set(df1['disposition']))

In [ ]:
df1['local_obs'].describe()

In [ ]:
df1['event_state'].describe()

In [ ]:
print(set(df1['event_state']))

In [ ]:
df1['sighting_week'].describe()

In [ ]:
df1['disposition_step'].describe()

In [ ]:
df1['use_sentiment'].describe()

In [ ]:
df1['use_sightings'].describe()

In [ ]:
QUERY = "SELECT * FROM x_post_sentiment"
df2 = pd.read_sql(QUERY, get_engine())

In [ ]:
df2

In [ ]:
sns.countplot(df1['lang'])
plt.title('Count of posts in different languages')
plt.ylabel('Languages')
plt.show()

In [ ]:
plt.figure(figsize=(10, 10))
sns.countplot(df1['event_state'], order=df1['event_state'].value_counts().index)
plt.title('Count of the posts states')
plt.ylabel('States in USA')
plt.show()

In [ ]:
sns.countplot(df2['lang'])
plt.title('Count of posts in different languages')
plt.ylabel('Languages')
plt.show()

In [ ]:
# df2 merge with df1 on post_id
df_sentiment_merge = df1.merge(df2, how='inner', on='post_id')

In [ ]:
df_sentiment_merge

In [ ]:
# stream says which of the four X post collections a row belongs to. 
# Each stream goes through the cleaning funnel separately, because the rules differ a 
# little by language and the English data source changes on 2025-01-01.

plt.figure(figsize=(10, 7))
sns.countplot(data=df_sentiment_merge, x='stream')
plt.title('Count of the Posts From Different Sources')
plt.ylabel('Count')
plt.xlabel('Sources (Prior From Previous Cohort)')
plt.show()

In [ ]:
# disposition records what happened to each row in the X cleaning funnel. 
# Every collected post goes through 15 filter steps in order, and disposition is the name of the first step that removed it.
# A row that passes every step gets "Sighting".

plt.figure(figsize=(10, 7))
sns.countplot(data=df_sentiment_merge, y='disposition', order=df_sentiment_merge['disposition'].value_counts().index)
plt.title('Count of the Filtering Reasons of the Records')
plt.ylabel('Count')
plt.xlabel('Dispositions')
plt.show()

In [ ]:
# use_sentiment
# is a True/False column that marks the filtered subset: the posts used to build sentiment features. 
# A row is True if it got through funnel steps 1–7. Those are the fixed, rule-based cleaning steps: 
# duplicates, date window, quote posts, no US state, foreign place names, hub authors, and copypasta repeats.

plt.figure(figsize=(10, 7))
sns.countplot(data=df_sentiment_merge, x='use_sentiment')
plt.title('Count of the Clean and Located in US Filtering Records')
plt.ylabel('Count')
plt.show()

In [ ]:
# use sighting

plt.figure(figsize=(10, 7))
sns.countplot(data=df_sentiment_merge, x='use_sightings')
plt.title('Count of the Records that Pass or Fail Filtering of a Firsthand Report of Sick or Dead Animals that Counts')
plt.ylabel('Count')
plt.show()

In [ ]:
# use sighting

plt.figure(figsize=(10, 7))
sns.countplot(data=df_sentiment_merge[df_sentiment_merge['use_sightings'] == True], x='use_sightings')
plt.title('Count of the Records that Pass Filtering of a Firsthand Report of Sick or Dead Animals that Counts')
plt.ylabel('Count')
plt.show()

# Sentiment Analysis

In [ ]:
df_sentiment_merge_enesfr = df_sentiment_merge[df_sentiment_merge['lang_x'].isin(['en', 'es', 'fr'])]

In [ ]:
# general sentiment
plt.figure(figsize=(10, 10))
sns.countplot(data=df_sentiment_merge, x='final_sentiment')
plt.title('Count of the Posts Sentiment in All Languges')
plt.ylabel('Count')
plt.xlabel('Posts Sentiment')
plt.show()

In [ ]:
plt.figure(figsize=(10, 10))
sns.countplot(data=df_sentiment_merge[df_sentiment_merge['use_sightings'] == True], x='final_sentiment')
plt.title('Count of the Posts Sentiment in All Languges (Passed filtering)')
plt.ylabel('Count')
plt.xlabel('Posts Sentiment')
plt.show()

In [ ]:
# general sentiment in each language
plt.figure(figsize=(10, 10))
sns.countplot(data=df_sentiment_merge_enesfr, x='lang_x', hue='final_sentiment')
plt.title('Count of the Posts Sentiment in Each Languges')
plt.ylabel('Count')
plt.xlabel('Posts Sentiment')
plt.show()


In [ ]:
import seaborn.objects as so

In [ ]:
top20_df = df_sentiment_merge_enesfr[df_sentiment_merge_enesfr["search_terms"].isin(list(top_n))]

In [ ]:
# general sentiment for top 20 search terms in each lan
p = (
    so.Plot(top20_df, x="lang_x", color="final_sentiment")
    .add(so.Bar(), so.Count(), so.Dodge())
    .facet(col="search_terms", wrap=5)
    .layout(size=(20, 14))
    .label(x="Language", y="Count", color="Sentiment")
)
p.show()

# Comparing Legitimate Sentiment with Ground Truth

In [ ]:
QUERY = "SELECT * FROM fact_h5n1_outbreak"
h5n1_df = pd.read_sql(QUERY, get_engine())

In [ ]:
h5n1_df

In [ ]:
# H5N1 birds affected in the United States 
plt.figure(figsize=(15, 10))
sns.lineplot(data=h5n1_df, x="day", y="birds_affected")
plt.title('H5N1 birds affected in the United States')
plt.ylabel('Bird Affected by H5N1 Count')
plt.xlabel('Date')
plt.show()

In [ ]:
df_filtered = df1[df1['use_sightings']==True]

In [ ]:
# H5N1 vs filtered sentiment
plt.figure(figsize=(15, 10))
sns.lineplot(data=df_filtered, x="created_at", y="sentiment")
plt.title('Sentiment of the United States Posts From 2022 to 2026')
plt.ylabel('Sentiment (positive_score − negative_score)')
plt.xlabel('Date Post')
plt.show()

In [ ]:
# Both x columns need to be real datetimes so the axes line up.
h5n1_df["day"] = pd.to_datetime(h5n1_df["day"])
df_filtered["created_at"] = pd.to_datetime(df_filtered["created_at"]).dt.tz_localize(None)

# Collapse to one value per day: total birds, mean sentiment.
birds_daily = h5n1_df.groupby("day", as_index=False)["birds_affected"].sum()
sent_daily = (df_filtered
              .groupby(df_filtered["created_at"].dt.floor("D"))["sentiment"]
              .mean()
              .rename_axis("day")
              .reset_index())

fig, ax1 = plt.subplots(figsize=(12, 5))

sns.lineplot(data=birds_daily, x="day", y="birds_affected", ax=ax1,
             color="tab:red", label="Birds affected")
ax1.set_ylabel("Birds affected", color="tab:red")

ax2 = ax1.twinx()   # second y-axis, same x-axis
sns.lineplot(data=sent_daily, x="day", y="sentiment", ax=ax2,
             color="tab:blue", label="Mean sentiment")
ax2.set_ylabel("Mean X sentiment (pos − neg)", color="tab:blue")
ax2.axhline(0, color="tab:blue", linewidth=0.5, linestyle="--")

# Each axis draws its own legend, so merge them into one.
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax2.legend(h1 + h2, l1 + l2, loc="upper left")
ax1.get_legend().remove()

ax1.set_xlabel("")
ax1.set_title("H5N1 birds affected vs. X sentiment")
plt.tight_layout()
plt.show()

# Export out the excel to Elda

In [ ]:
df_sentiment_merge_fr = df_sentiment_merge[df_sentiment_merge['lang_x'] == 'fr']
df_sentiment_merge_es = df_sentiment_merge[df_sentiment_merge['lang_x'] == 'es']

In [ ]:
df_sentiment_merge_fr

In [ ]:
df_sentiment_merge_es

In [ ]:
# sample from every search terms, sample for 100 posts each languagues

def sample_lan(df,lan, num = 100):
    sample_df = df.sample(n=num)
    new_sample = sample_df.loc[:,["post_id","cleaned_post_text","negative_score_y","neutral_score_y","positive_score_y","final_sentiment"]]
    new_sample["Agree?(Y/N)"] = ""
    new_sample["Correct Sentiment"] = ""
    new_sample.to_excel(f'sentiment_sample_{lan}.xlsx', index=False)
    return new_sample

In [ ]:
sample_fr = sample_lan(df_sentiment_merge_fr, lan="fr")

In [ ]:
sample_fr

In [ ]:
sample_es = sample_lan(df_sentiment_merge_es, lan="es")

In [ ]:
sample_es

# Emoji vs. No Emoji Sentiment

In [ ]:
emoji_df = pd.read_sql("SELECT * FROM x_post_noemoji_sentiment;", get_engine())

In [ ]:
emoji_df

In [ ]:
df2

In [ ]:
emoji_vs_no_df = df2.merge(emoji_df.loc[:,["post_id","no_emoji_negative_score","no_emoji_neutral_score","no_emoji_positive_score","no_emoji_final_sentiment"]], how='inner', on='post_id')

In [ ]:
emoji_vs_no_df

In [ ]:
# Keep only posts where removing emojis changed the predicted sentiment label
sentiment_changed_df = emoji_vs_no_df[
    emoji_vs_no_df["final_sentiment"] != emoji_vs_no_df["no_emoji_final_sentiment"]
].copy()

print(f"{len(sentiment_changed_df):,} of {len(emoji_vs_no_df):,} posts "
      f"({len(sentiment_changed_df) / len(emoji_vs_no_df):.1%}) changed sentiment")
sentiment_changed_df

In [ ]:
plt.figure(figsize=(10, 10))
sns.countplot(data=sentiment_changed_df, x='final_sentiment', hue='no_emoji_final_sentiment', order=sentiment_changed_df['no_emoji_final_sentiment'].value_counts().index)
plt.show()

In [ ]:
pd.crosstab(sentiment_changed_df["final_sentiment"],
            sentiment_changed_df["no_emoji_final_sentiment"]).plot(
    kind="bar", stacked=True, figsize=(10, 6))
plt.xlabel("Sentiment with emoji")
plt.ylabel("Number of posts")
plt.legend(title="Sentiment without emoji")
plt.title("Sentiment Changes After Emoji Removal")
plt.xticks(rotation=0)
plt.show()

# Meta Post Analysis

In [ ]:
QUERY = "SELECT * FROM meta_post_unique LIMIT 10"
meta_unique_df = pd.read_sql(QUERY, get_engine())

In [ ]:
meta_unique_df

In [ ]:
QUERY = "SELECT * FROM meta_post_scored LIMIT 10"
meta_scored_df = pd.read_sql(QUERY, get_engine())

In [ ]:
meta_scored_df

kw_hits is  the number of times an H5N1 search keyword appears in the post's text.

In [ ]:
QUERY = "SELECT * FROM meta_post_analysis LIMIT 10"
meta_analysis_df = pd.read_sql(QUERY, get_engine())

In [ ]:
meta_analysis_df

What is the geo_tier and use sentiment?

In [ ]:
QUERY = "SELECT * FROM  meta_post_funnel LIMIT 10"
meta_funnel_df = pd.read_sql(QUERY, get_engine())

In [ ]:
meta_funnel_df

In [ ]:
QUERY = "SELECT * FROM  meta_post_sentiment LIMIT 10"
meta_sentiment_df = pd.read_sql(QUERY, get_engine())

In [ ]:
meta_sentiment_df

In [ ]:
QUERY = "SELECT * FROM meta_post_funnel"
meta_funnel_df = pd.read_sql(QUERY, get_engine())

In [ ]:
meta_funnel_df.info()

In [ ]:
sns.countplot(data=meta_funnel_df, x='content_type', order= meta_funnel_df['content_type'].value_counts().index)
plt.title('Content types in Meta Posts')
plt.ylabel('Post Counts')
plt.xlabel('Content Types')
plt.show()

In [ ]:
sns.countplot(data=meta_funnel_df, x='use_sightings', order= meta_funnel_df['use_sightings'].value_counts().index)
plt.title('Count of the Posts Sentiment in All Languges (Passed filtering) of Meta Posts')
plt.ylabel('Post Counts')
plt.xlabel('Use Sightings')
plt.show()

In [ ]:
sns.countplot(data=meta_funnel_df, x='use_sentiment', order= meta_funnel_df['use_sentiment'].value_counts().index)
plt.title('Count of Clean Enough Post for Sentiment in Meta Posts')
plt.ylabel('Post Counts')
plt.xlabel('Use Sentiment')
plt.show()

In [ ]:
sns.countplot(data=meta_funnel_df, y='disposition', order= meta_funnel_df['disposition'].value_counts().index)
plt.title('Dispositions for Filtering Out Posts in Meta')
plt.ylabel('Reasons For Filtering Posts')
plt.xlabel('Post Counts')
plt.show()

In [ ]:
sns.countplot(data=meta_funnel_df, x='disposition_step')
plt.title('Dispositions Steps in Meta')
plt.ylabel('Post Counts')
plt.xlabel('Disposition Steps')
plt.show()

In [ ]:
plt.figure(figsize=(10, 10))
sns.countplot(data=meta_funnel_df, y='event_state', order=meta_funnel_df['event_state'].value_counts().index)
plt.title('Meta Post Counts by States')
plt.xlabel('Post Counts')
plt.ylabel('States')
plt.show()

In [ ]:
sns.countplot(data=meta_funnel_df, x='exclusion_reason', order=meta_funnel_df['exclusion_reason'].value_counts().index)
plt.title('Exclusion Reasons for Posts in Meta')
plt.ylabel('Post Counts')
plt.xlabel('Exclusion Reasons')
plt.show()

In [ ]:
sns.countplot(data=meta_funnel_df, x='owner_type', order=meta_funnel_df['owner_type'].value_counts().index)
plt.title('Owner Type of Post in Meta')
plt.ylabel('Post Counts')
plt.xlabel('Owner Type')
plt.show()

In [ ]:
sns.countplot(data=meta_funnel_df, x='account_type', order=meta_funnel_df['account_type'].value_counts().index)
plt.title('Account Type of Post in Meta')
plt.ylabel('Post Counts')
plt.xlabel('Account Type')
plt.show()

In [ ]:
sns.countplot(data=meta_funnel_df, x='hub', order=meta_funnel_df['hub'].value_counts().index)
plt.title('Hub Account of Post in Meta')
plt.ylabel('Post Counts')
plt.xlabel('Hub Account (True or False)')
plt.show()

In [ ]:
sns.countplot(data=meta_funnel_df, y='search_terms', order=meta_funnel_df['search_terms'].value_counts().iloc[:20].index)
plt.title('Top 20 Search Terms of Post in Meta')
plt.ylabel('Search Terms')
plt.xlabel('Post Counts')
plt.show()